# Chapter 11 Companion Notebook: High-Frequency Trading

This notebook reproduces every worked numerical example from Chapter 11 of *AI in Finance* (`chapter11.tex`): the fiber-vs-microwave latency calculation, the latency-arbitrage worked example, the Glosten-Milgrom sequential-trade model and PIN, ETF-basket statistical arbitrage, the high-frequency extension of the Avellaneda-Stoikov inventory model, daily market-making P&L, cross-market (futures-leading-cash) latency arbitrage, the co-location break-even calculation, order-to-trade ratio fees, effective spreads, the order-flow-imbalance (OFI) regression, and the queue-position (price-time priority) example. Every number below matches the corresponding number in the chapter text exactly.

## 1. Fiber vs. microwave latency (Section: The Race to Zero)

In [1]:

distance_km = 1200
c = 299_792.458  # km/s, speed of light in vacuum

fiber_speed = 0.68 * c
microwave_speed = 0.9997 * c

t_fiber_ms = (2 * distance_km / fiber_speed) * 1000
t_microwave_ms = (2 * distance_km / microwave_speed) * 1000

print(f"Fiber round-trip: {t_fiber_ms:.2f} ms")
print(f"Microwave round-trip: {t_microwave_ms:.2f} ms")
print(f"Advantage: {t_fiber_ms - t_microwave_ms:.2f} ms")


Fiber round-trip: 11.77 ms
Microwave round-trip: 8.01 ms
Advantage: 3.76 ms


## 2. Latency arbitrage: stale quotes (Section: A Worked Example, latency arbitrage)

In [2]:
price_jump = 0.05
shares_per_event = 5000
events_per_day = 20

profit_per_event = shares_per_event * price_jump
daily_profit = events_per_day * profit_per_event

print(f"Profit per event: ${profit_per_event:,.2f}")
print(f"Daily profit: ${daily_profit:,.2f}")


Profit per event: $250.00
Daily profit: $5,000.00


## 3. Glosten-Milgrom sequential-trade model (Section: A More Advanced Model)

In [3]:

V_H, V_L = 51.0, 49.0
pi = 0.10  # fraction of informed traders

def quote_round(prior_H):
    prior_L = 1 - prior_H
    p_buy_given_H = pi * 1.0 + (1 - pi) * 0.5
    p_buy_given_L = pi * 0.0 + (1 - pi) * 0.5
    p_buy = prior_H * p_buy_given_H + prior_L * p_buy_given_L

    p_sell_given_H = (1 - pi) * 0.5
    p_sell_given_L = pi * 1.0 + (1 - pi) * 0.5
    p_sell = prior_H * p_sell_given_H + prior_L * p_sell_given_L

    post_H_given_buy = (p_buy_given_H * prior_H) / p_buy
    post_H_given_sell = (p_sell_given_H * prior_H) / p_sell

    ask = post_H_given_buy * V_H + (1 - post_H_given_buy) * V_L
    bid = post_H_given_sell * V_H + (1 - post_H_given_sell) * V_L

    return {'prior_H': prior_H, 'post_H_given_buy': post_H_given_buy,
            'post_H_given_sell': post_H_given_sell, 'ask': ask, 'bid': bid}

r1 = quote_round(0.5)
print("Round 1 (prior 0.5):", {k: round(v, 4) for k, v in r1.items()})
print(f"Spread round 1: ${r1['ask'] - r1['bid']:.2f}")
print(f"Check pi*(V_H-V_L): ${pi * (V_H - V_L):.2f}")

r2 = quote_round(r1['post_H_given_buy'])
print("\nRound 2 (prior = post_H_given_buy from round 1, after a BUY):",
      {k: round(v, 4) for k, v in r2.items()})
print(f"Spread round 2: ${r2['ask'] - r2['bid']:.2f}")


Round 1 (prior 0.5): {'prior_H': 0.5, 'post_H_given_buy': 0.55, 'post_H_given_sell': 0.45, 'ask': 50.1, 'bid': 49.9}
Spread round 1: $0.20
Check pi*(V_H-V_L): $0.20

Round 2 (prior = post_H_given_buy from round 1, after a BUY): {'prior_H': 0.55, 'post_H_given_buy': 0.599, 'post_H_given_sell': 0.5, 'ask': 50.198, 'bid': 50.0}
Spread round 2: $0.20


## 4. Probability of Informed Trading, PIN (Section 11's PIN subsection)

In [4]:

alpha_event = 0.30
mu = 150
epsilon = 100

pin = (alpha_event * mu) / (alpha_event * mu + 2 * epsilon)
print(f"alpha*mu = {alpha_event*mu}")
print(f"2*epsilon = {2*epsilon}")
print(f"PIN = {pin:.4f} ({pin:.1%})")


alpha*mu = 45.0
2*epsilon = 200
PIN = 0.1837 (18.4%)


## 5. ETF-basket statistical arbitrage (Section: Statistical Arbitrage at High Frequency)

In [5]:

etf_price = 150.05
basket_fair_value = 150.00
mispricing = etf_price - basket_fair_value
mispricing_pct = mispricing / basket_fair_value

position = 10_000_000
profit_per_capture = position * mispricing_pct

events_per_day = 100
avg_position_per_event = 200_000
profit_per_event = avg_position_per_event * mispricing_pct
daily_profit = events_per_day * profit_per_event

print(f"Mispricing: ${mispricing:.2f} ({mispricing_pct:.4%})")
print(f"Profit on $10M position: ${profit_per_capture:,.2f}")
print(f"Profit per event (avg $200k position): ${profit_per_event:,.2f}")
print(f"Daily profit (100 events/day): ${daily_profit:,.2f}")


Mispricing: $0.05 (0.0333%)
Profit on $10M position: $3,333.33
Profit per event (avg $200k position): $66.67
Daily profit (100 events/day): $6,666.67


## 6. High-frequency extension of the Avellaneda-Stoikov inventory model (Section: Market Making at High Frequency)

In [6]:

q, gamma, sigma = 5, 0.1, 2

def skew(T_minus_t):
    return q * gamma * sigma**2 * T_minus_t

skew_full_day = skew(1)
skew_hf = skew(0.001)

print(f"skew(T-t=1) = {skew_full_day:.2f}")
print(f"skew(T-t=0.001) = {skew_hf:.3f}")
print(f"Reduction factor: {skew_full_day/skew_hf:.0f}x")


skew(T-t=1) = 2.00
skew(T-t=0.001) = 0.002
Reduction factor: 1000x


## 7. Daily P&L from high-frequency market making (Section: A Worked Example, daily P&L)

In [7]:

round_trips = 50_000
avg_size = 100
half_spread = 0.005

gross_spread_capture = round_trips * avg_size * half_spread

toxic_fraction = 0.05
toxic_loss_per_share = 0.02
adverse_selection_loss = (round_trips * toxic_fraction) * avg_size * toxic_loss_per_share

net_profit = gross_spread_capture - adverse_selection_loss

print(f"Gross spread capture: ${gross_spread_capture:,.2f}")
print(f"Adverse selection loss: ${adverse_selection_loss:,.2f}")
print(f"Net daily profit: ${net_profit:,.2f}")
print(f"Adverse selection as % of gross: {adverse_selection_loss/gross_spread_capture:.1%}")


Gross spread capture: $25,000.00
Adverse selection loss: $5,000.00
Net daily profit: $20,000.00
Adverse selection as % of gross: 20.0%


## 8. Cross-market latency arbitrage: futures leading the cash market (Section: Cross-Market Latency Arbitrage)

In [8]:

futures_move_points = 2.00
etf_tracking_ratio = 1/10
implied_etf_move = futures_move_points * etf_tracking_ratio

shares = 1_000
profit_per_event = shares * implied_etf_move

events_per_day = 30
daily_profit = events_per_day * profit_per_event

print(f"Implied ETF fair-value move: ${implied_etf_move:.2f}")
print(f"Profit per event: ${profit_per_event:,.2f}")
print(f"Daily profit: ${daily_profit:,.2f}")


Implied ETF fair-value move: $0.20
Profit per event: $200.00
Daily profit: $6,000.00


## 9. Co-location break-even calculation (Section: The Economics of Co-location)

In [9]:

colo_fee = 25_000
data_fee = 10_000
total_monthly_cost = colo_fee + data_fee

profit_per_event = 250
trading_days_per_month = 21

breakeven_events_per_month = total_monthly_cost / profit_per_event
breakeven_events_per_day = breakeven_events_per_month / trading_days_per_month

assumed_events_per_day = 20
assumed_monthly_profit = assumed_events_per_day * trading_days_per_month * profit_per_event

print(f"Total monthly cost: ${total_monthly_cost:,.0f}")
print(f"Break-even events needed per month: {breakeven_events_per_month:.1f}")
print(f"Break-even events needed per day: {breakeven_events_per_day:.2f}")
print(f"Assumed monthly profit at 20 events/day: ${assumed_monthly_profit:,.0f}")
print(f"Multiple of break-even: {assumed_monthly_profit/total_monthly_cost:.1f}x")


Total monthly cost: $35,000
Break-even events needed per month: 140.0
Break-even events needed per day: 6.67
Assumed monthly profit at 20 events/day: $105,000
Multiple of break-even: 3.0x


## 10. Order-to-trade ratio and message fees (Section: Order-to-Trade Ratios)

In [10]:

messages = 2_000_000
trades = 40_000
order_to_trade_ratio = messages / trades

allowed_ratio = 20
fee_per_excess_message = 0.01

allowed_messages = trades * allowed_ratio
excess_messages = messages - allowed_messages
total_fee = excess_messages * fee_per_excess_message

print(f"Order-to-trade ratio: {order_to_trade_ratio:.0f}:1")
print(f"Allowed messages: {allowed_messages:,.0f}")
print(f"Excess messages: {excess_messages:,.0f}")
print(f"Total fee: ${total_fee:,.2f}")


Order-to-trade ratio: 50:1
Allowed messages: 800,000
Excess messages: 1,200,000
Total fee: $12,000.00


## 11. Effective spread (Section: Measuring Market Quality)

In [11]:

midpoint = (50.07 + 50.08) / 2

exec_at_ask = 50.08
effective_spread_no_improvement = 2 * abs(exec_at_ask - midpoint)

exec_with_improvement = 50.078
effective_spread_improved = 2 * abs(exec_with_improvement - midpoint)

print(f"Midpoint: ${midpoint:.3f}")
print(f"Effective spread (execution at ask): ${effective_spread_no_improvement:.3f}")
print(f"Effective spread (execution with price improvement): ${effective_spread_improved:.3f}")


Midpoint: $50.075
Effective spread (execution at ask): $0.010
Effective spread (execution with price improvement): $0.006


## 12. Order flow imbalance (OFI) regression (Section: Predictive Modeling, OFI)

In [12]:

import numpy as np

ofi = np.array([50, -30, 80, -60, 20, 40], dtype=float)
dprice_ticks = np.array([2, -1, 3, -2, 1, 2], dtype=float)  # cents

xbar, ybar = ofi.mean(), dprice_ticks.mean()
beta = np.sum((ofi - xbar) * (dprice_ticks - ybar)) / np.sum((ofi - xbar) ** 2)
alpha = ybar - beta * xbar
fitted = alpha + beta * ofi
resid = dprice_ticks - fitted
r2 = 1 - np.sum(resid**2) / np.sum((dprice_ticks - ybar) ** 2)

print(f"alpha = {alpha:.4f}")
print(f"beta = {beta:.5f}")
print(f"R^2 = {r2:.4f}")
print(f"Predicted price change for OFI=65: {alpha + beta*65:.3f} cents")


alpha = 0.2184
beta = 0.03689
R^2 = 0.9925
Predicted price change for OFI=65: 2.617 cents


## 13. Queue position and price-time priority (Section: Queue Position and the Economics of Price-Time Priority)

In [13]:

shares_ahead = 400
execution_rate_per_sec = 10  # 600 shares/minute

expected_wait_ahead = shares_ahead / execution_rate_per_sec

own_order_size = 100
expected_wait_own_fill = (shares_ahead + own_order_size) / execution_rate_per_sec

new_queue_ahead_after_cancel = 250
expected_wait_after_cancel = new_queue_ahead_after_cancel / execution_rate_per_sec

print(f"Expected wait until queue ahead clears: {expected_wait_ahead:.0f} seconds")
print(f"Expected wait until own order fully filled: {expected_wait_own_fill:.0f} seconds")
print(f"Expected wait after cancel/resubmit (new queue): {expected_wait_after_cancel:.0f} seconds")


Expected wait until queue ahead clears: 40 seconds
Expected wait until own order fully filled: 50 seconds
Expected wait after cancel/resubmit (new queue): 25 seconds
